# Interim Dataset Validation

This notebook validates that the interim kidney disease dataset was generated correctly by the `src/perform_minimal_data_cleaning.py` script.

## 1. Load the Interim Dataset

Load the interim dataset from the data directory using pandas. Display basic information about the loaded dataset.

In [ ]:
import pandas as pd
from pathlib import Path

# Load interim dataset
interim_path = Path("../data/interim/kidney_disease_interim.csv")
df_interim = pd.read_csv(interim_path)

print("Interim dataset loaded successfully!")
print(f"\nDataset info:")
print(f"  Path: {interim_path}")
print(f"  File exists: {interim_path.exists()}")
print(f"\nFirst few rows:")
print(df_interim.head())

## 2. Check Dataset Shape

Verify the shape of the interim dataset (number of rows and columns).

In [ ]:
# Load raw dataset to compare
raw_path = Path("../data/raw/kidney_disease.csv")
df_raw = pd.read_csv(raw_path, header=0, skiprows=[1, 2])

# Columns removed during cleaning (data leakage)
LEAKAGE_COLUMNS = {'affected', 'stage'}

# Check shapes
interim_shape = df_interim.shape
raw_shape = df_raw.shape

# Expected: 1 row removed ('p' in grf), 2 leakage columns removed
shape_rows_match = interim_shape[0] == raw_shape[0] - 1
shape_cols_match = interim_shape[1] == raw_shape[1] - len(LEAKAGE_COLUMNS)

print("Dataset Shape Validation")
print(f"{'='*50}")
print(f"Interim dataset shape: {interim_shape[0]} rows × {interim_shape[1]} columns")
print(f"Raw dataset shape (after removing metadata): {raw_shape[0]} rows × {raw_shape[1]} columns")
print(f"\nRow count correct (1 invalid row removed): {shape_rows_match}")
print(f"Column count correct ({len(LEAKAGE_COLUMNS)} leakage columns removed): {shape_cols_match}")

if interim_shape[0] > 0:
    print(f"Data rows present: {interim_shape[0]} rows")
if interim_shape[1] > 0:
    print(f"Columns present: {interim_shape[1]} columns")


## 3. Verify Metadata Removal

Check for any remaining metadata rows or columns that shouldn't be present.

In [ ]:
# Read raw file as-is to show metadata
with open(raw_path, 'r') as f:
    raw_lines = f.readlines()[:4]

print("Metadata Removal Verification")
print(f"{'='*50}")
print("\nRaw file first 3 lines (with metadata):")
for i, line in enumerate(raw_lines[:3], 1):
    print(f"  Line {i}: {line.strip()[:80]}...")

print("\n\nInterim dataset - checking for metadata indicators:")
metadata_indicators = ['discrete', 'class', 'meta']
found_metadata = False

for indicator in metadata_indicators:
    count = (df_interim == indicator).sum().sum()
    if count > 0:
        print(f"  Found '{indicator}': {count} occurrences!")
        found_metadata = True

if not found_metadata:
    print("  No metadata indicators found in interim dataset!")

## 4. Validate Column Names

Compare the column names of the interim dataset with the raw (cleaned) dataset.

In [ ]:
# Compare column names
interim_cols = set(df_interim.columns)
raw_cols = set(df_raw.columns)

# Expected interim columns: raw columns minus leakage columns
expected_interim_cols = raw_cols - LEAKAGE_COLUMNS

print("Column Names Validation")
print(f"{'='*50}")

# Check that leakage columns are absent
leakage_present = LEAKAGE_COLUMNS & interim_cols
leakage_absent = len(leakage_present) == 0

# Check that all remaining columns match
cols_match = interim_cols == expected_interim_cols

print(f"\nLeakage columns absent ('affected', 'stage'): {leakage_absent}")
if leakage_present:
    print(f"  Still present: {leakage_present}")

print(f"All non-leakage columns present: {cols_match}")
print(f"  Interim columns: {len(interim_cols)}")
print(f"  Expected columns: {len(expected_interim_cols)}")

if not cols_match:
    missing = expected_interim_cols - interim_cols
    extra = interim_cols - expected_interim_cols
    if missing:
        print(f"\nMissing columns: {missing}")
    if extra:
        print(f"\nExtra columns: {extra}")
else:
    print(f"\nAll expected columns present!")


## 5. Generate Validation Report

Create a comprehensive validation report summarizing all checks.

In [ ]:
# Generate comprehensive validation report
print("INTERIM DATASET VALIDATION REPORT")
print("="*60)

# Prepare validation status
checks = {
    "Dataset loaded successfully": interim_path.exists(),
    "Row count correct (1 invalid row removed)": shape_rows_match,
    "Column count correct (2 leakage columns removed)": shape_cols_match,
    "Data rows present": interim_shape[0] > 0,
    "Columns present": interim_shape[1] > 0,
    "Leakage columns absent ('affected', 'stage')": leakage_absent,
    "All non-leakage columns present": cols_match,
    "No metadata indicators": not found_metadata,
}

# Print validation results
print(f"\nVALIDATION SUMMARY:")
print("-" * 60)

passed = 0
failed = 0

for check_name, status in checks.items():
    result = "[OK]" if status else "[NOK]"
    print(f"  {result} {check_name}")
    if status:
        passed += 1
    else:
        failed += 1

print("-" * 60)
print(f"\nResults: {passed}/{len(checks)} checks passed")

# Overall validation status
if failed == 0:
    print("\nSUCCESS: Interim dataset is VALID!")
    print("\nThe dataset was cleaned correctly:")
    print(f"  • Shape: {interim_shape[0]} rows × {interim_shape[1]} columns")
    print(f"  • Invalid row ('p' in grf) removed successfully")
    print(f"  • Leakage columns ('affected', 'stage') removed")
    print(f"  • All remaining columns preserved from original dataset")
else:
    print(f"\nWARNING: {failed} validation check(s) failed!")
    print("Please review the dataset and cleaning script.")

print("\n" + "="*60)
